In [1]:
import numpy as np
import pandas as pd
import random

import re
from collections import Counter
import matplotlib.pyplot as plt

# Data Understanding

In [2]:
series_data = pd.read_csv("web_series.csv")
series_data.head(15)

,Series Title,Year Released,Content Rating,IMDB Rating,R Rating,Genre,Description,No of Seasons,Streaming Platform
0,Breaking Bad,2008,18+,9.5,100,"Crime,Drama","When Walter White, a New Mexico chemistry teac...",5Seasons,Netflix
1,Game of Thrones,2011,18+,9.3,99,"Action & Adventure,Drama",Seven noble families fight for control of the ...,8Seasons,"HBO MAX,HBO"
2,Rick and Morty,2013,18+,9.2,97,"Animation,Comedy",Rick is a mentally-unbalanced but scientifical...,4Seasons,"Free Services,HBO MAX,Hulu"
3,Stranger Things,2016,16+,8.8,96,"Drama,Fantasy","When a young boy vanishes, a small town uncove...",3Seasons,Netflix
4,The Boys,2019,18+,8.7,95,"Action & Adventure,Comedy",A group of vigilantes known informally as “The...,2Seasons,Prime Video
5,Dark,2017,16+,8.8,95,"Crime,Drama",A missing child causes four families to help e...,3Seasons,Netflix
6,Chernobyl,2019,18+,9.4,95,"Drama,History",The true story of one of the worst man-made ca...,1Season,"HBO MAX,HBO"
7,Sherlock,2010,16+,9.1,94,"Action & Adventure,Crime",A modern update finds the famous sleuth and hi...,4Seasons,Netflix
8,Fargo,2014,18+,8.9,94,"Crime,Drama",A close-knit anthology series dealing with sto...,3Seasons,Hulu
9,Avatar: The Last Airbender,2005,7+,9.2,94,"Action & Adventure,Animation","In a war-torn world of elemental magic, a youn...",3Seasons,"Netflix,CBS All Access,Hoopla"


In [3]:
series_data.sample(10)

,Series Title,Year Released,Content Rating,IMDB Rating,R Rating,Genre,Description,No of Seasons,Streaming Platform
7946,Pororo the Little Penguin,2003,all,5.7,39,"Animation,Children",Pororo the Little Penguin is a computer-genera...,6Seasons,"Netflix,Prime Video,Hulu"
11781,Second World War Diary (1939-1945),2009,NaN,NaN,-1,-1,-1,-1,-1
11339,Monsters in My Head,2012,NaN,NaN,10,-1,-1,-1,-1
7243,Chip and Potato,2018,all,6.8,42,"Children,Family","Lovable pug Chip starts kindergarten, makes ne...",2Seasons,Netflix
4813,Street Fighter: Resurrection,2016,16+,6.5,51,"Action & Adventure,Crime",Taking place ten years after Street Fighter: A...,1Season,Prime Video
9638,Maharakshak Devi,2015,NaN,6.1,30,"Fantasy,2015",Devi is the story of the most powerful girl in...,1Season,Netflix
8545,Blackbeard,2006,NaN,6.3,36,"Documentary,Drama","Drama documentary about Edward Teach, also kno...",1 Season,"Free Services,Hoopla"
8813,Murder Decoded,2019,16+,7.3,35,"Action & Adventure,Comedy","In the wake of every murder, clues appear. Mur...",1Season,fuboTV
441,Da Vinci's Demons,2013,18+,8.0,78,"Action & Adventure,Drama","The series follows the ""untold"" story of Leona...",3Seasons,Starz
7578,Inside the FBI: New York,2017,18+,7.0,41,"Documentary,Reality",An unprecedented look at the FBI's New York fi...,1Season,USA


In [4]:
series_data.shape

(12353, 9)

In [5]:
series_data.isnull().mean()*100

Series Title           0.000000
Year Released          0.000000
Content Rating        41.455517
IMDB Rating           17.372298
R Rating               0.000000
Genre                  0.000000
Description            0.000000
No of Seasons          0.000000
Streaming Platform    16.052781
dtype: float64

In [6]:
series_data.describe()

,Year Released,IMDB Rating,R Rating
count,12353.000000,10207.000000,12353.000000
mean,2010.495345,6.973156,43.982029
std,11.240943,1.149726,20.660914
min,1901.000000,1.000000,-1.000000
25%,2009.000000,6.400000,33.000000
50%,2014.000000,7.200000,46.000000
75%,2017.000000,7.800000,58.000000
max,2020.000000,9.700000,100.000000


In [7]:
series_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12353 entries, 0 to 12352
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Series Title        12353 non-null  object 
 1   Year Released       12353 non-null  int64  
 2   Content Rating      7232 non-null   object 
 3   IMDB Rating         10207 non-null  float64
 4   R Rating            12353 non-null  int64  
 5   Genre               12353 non-null  object 
 6   Description         12353 non-null  object 
 7   No of Seasons       12353 non-null  object 
 8   Streaming Platform  10370 non-null  object 
dtypes: float64(1), int64(2), object(6)
memory usage: 868.7+ KB


In [8]:
series_data.duplicated().sum()

0

# Data Cleaning

In [9]:
series_data.sample(10)

,Series Title,Year Released,Content Rating,IMDB Rating,R Rating,Genre,Description,No of Seasons,Streaming Platform
8210,The Magicians,2011,NaN,7.9,38,"Reality,Comedy",The Magicians was a British family entertainme...,2 Seasons,Free Services
7699,Mickey's 90th Spectacular,2018,7+,7.0,40,"Family,Children","The televised special, celebrating the 90th bi...",1Season,Hulu
1878,Cranford,2007,7+,8.3,64,"Drama,2007",A rich and comic drama about the people of Cra...,2 Seasons,BritBox
10479,Ms. Adventure,2007,NaN,6.7,22,"Documentary,2007",Ms. Adventure was an American documentary real...,1Season,NaN
699,Queer as Folk,1999,18+,8.2,74,"Drama,LGBTQ",The lives and loves of young men in and around...,5Seasons,"Free Services,Prime Video,Showtime"
2153,The Last Post,2017,18+,7.2,63,"Drama,History","Drama series set in the mid-sixties, in which ...",1Season,Prime Video
9562,My Crazy Obsession,2012,NaN,6.1,31,"Reality,Documentary",My Crazy Obsession pulls back the curtains to ...,2Seasons,TLC
10363,Chiquis 'n Control,2012,NaN,7.4,24,"2012,Free Services",Chiquis heads to New York City and confronts h...,1Season,"Free Services,Hulu"
8866,Robson Green: Extreme Fisherman,2014,NaN,7.5,35,"Documentary,2014","In a brand new series for Quest, Robson Green ...",1Season,NaN
10920,Awesome Animals,2020,NaN,NaN,15,"2020,Free Services",National Geographic presents a natural history...,1Season,"Free Services,Disney+"


In [10]:
def clean_genres(genres_str):
    # Remove numeric values (likely years)
    genres_str = re.sub(r'\b\d{4}\b', '', genres_str)
    # Remove extra whitespace and split by commas
    genres_list = [genre.strip() for genre in genres_str.split(',') if genre.strip()]
    return genres_list

In [11]:
series_data['genre'] = series_data['Genre'].apply(clean_genres)

In [12]:
series_data.sample(2)

,Series Title,Year Released,Content Rating,IMDB Rating,R Rating,Genre,Description,No of Seasons,Streaming Platform,genre
4695,The Millers,2013,16+,5.9,52,"Comedy,2013","A divorced reporter, looking forward to the si...",2Seasons,NaN,[Comedy]
9694,Between the Sky and Sea,2018,16+,4.8,30,"Animation,Action & Adventure","Set in Onomichi City, six girls dream of becom...",1Season,"Free Services,Crunchyroll","[Animation, Action & Adventure]"


In [13]:
# Display the unique genres
unique_genres = set([genre for genres_list in series_data['genre'] for genre in genres_list])
print(unique_genres)

{'Romance', 'Documentary', 'Viceland', 'Apple TV+', 'FYI', 'Anime', 'Free Services', 'Disney+', 'NatGeo', 'BritBox', 'Action & Adventure', 'History', 'HGTV', 'Crime', 'Biography', 'Drama', 'Hulu', 'Comedy Central', 'Home & Garden', 'Food', 'Funimation', 'Lifetime', 'BET+', 'Science-Fiction', 'Travel Channel', 'Thriller', 'Disney', 'Fantasy', 'Travel', 'Hoopla', 'DIY', 'Reality', 'Prime Video', 'LGBTQ', 'Netflix', 'Horror', 'Showtime', 'AMC', 'Sport', 'Family', 'Children', 'fuboTV', 'HBO MAX', 'FOX', 'Game Show', '-1', 'Mystery', 'Comedy', 'MTV', 'BET', 'AcornTV', 'Cinemax', 'Pet', 'Food Network', 'VH1', 'CBS All Access', 'HBO', 'A&E', 'Bravo', 'Cult', 'Science', 'Musical', 'TLC', 'Starz', 'DC Universe', 'Animation', 'Stand-up & Talk'}


In [14]:
# Define valid genres
valid_genres = {
    'Action & Adventure', 'Comedy', 'Drama', 'Fantasy', 'Thriller', 'Mystery', 'Horror', 
    'Science-Fiction', 'Documentary', 'Animation', 'Romance', 'Musical', 'Family', 
    'Crime', 'Biography', 'Sport', 'Game Show', 'Reality', 'Stand-up & Talk', 'Children'
}

# Filter and clean the genre column
series_data['genre'] = series_data['genre'].apply(lambda genres: [g for g in genres if g in valid_genres])

# Remove rows with empty genre lists
series_data = series_data[series_data['genre'].map(len) > 0]

In [15]:
# Display the unique genres
unique_genres = set([genre for genres_list in series_data['genre'] for genre in genres_list])
print(unique_genres)

{'Romance', 'Documentary', 'Action & Adventure', 'Crime', 'Biography', 'Drama', 'Science-Fiction', 'Thriller', 'Fantasy', 'Reality', 'Horror', 'Sport', 'Family', 'Children', 'Game Show', 'Mystery', 'Comedy', 'Musical', 'Animation', 'Stand-up & Talk'}


In [16]:
series_data.shape

(10378, 10)

In [17]:
series_data.sample(4)

,Series Title,Year Released,Content Rating,IMDB Rating,R Rating,Genre,Description,No of Seasons,Streaming Platform,genre
1100,Airwolf,1984,7+,6.7,70,"Action & Adventure,Science-Fiction",Airwolf is an American television series that ...,4Seasons,Free Services,"[Action & Adventure, Science-Fiction]"
3068,The Spoils of Babylon,2014,16+,6.8,58,"Comedy,2014",The Spoils of Babylon is an American comedy mi...,2 Seasons,IFC,[Comedy]
833,Terriers,2010,18+,8.4,72,"Drama,Comedy",Ex-cop and recovering alcoholic Hank Dolworth ...,1Season,Hulu,"[Drama, Comedy]"
9586,Christmas Through the Decades,2015,NaN,7.9,31,"Family,Documentary",Take a trip back in time to see what Christmas...,1Season,Prime Video,"[Family, Documentary]"


In [18]:
series_data.shape

(10378, 10)

In [19]:
series_data_genre_cleaned = series_data[["Series Title", "genre", "Description", "Streaming Platform"]]
series_data_genre_cleaned.rename(columns={
    'Series Title': 'series_title',
    'genre': 'genres',
    'Description': 'description',
    'Streaming Platform': 'platform'
}, inplace=True)

/tmp/ipykernel_11833/2010518312.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  series_data_genre_cleaned.rename(columns={


In [20]:
series_data_genre_cleaned.sample(5)

,series_title,genres,description,platform
9090,Goodbye Dear Wife,"[Drama, Romance]",Goodbye Dear Wife is a 2012 South Korean telev...,Netflix
8177,Noddy's Toyland Adventures,"[Animation, Family]","It follows the adventures of Noddy, a little w...",Free Services
8855,Minami Kamakura High School Girls Cycling Club,[Animation],"Maiharu Hiromi has moved to Kamakura Nagasaki,...",Free Services
4797,Human Weapon,"[Documentary, Reality]",Human Weapon was a television show on The Hist...,"Free Services,History"
3635,The Ray Bradbury Theater,"[Drama, Fantasy]",A Canadian-produced fantastic anthology series...,"Free Services,Prime Video,Hoopla"


In [21]:
series_data_genre_cleaned['platform'].unique()

array(['Netflix', 'HBO MAX,HBO', 'Free Services,HBO MAX,Hulu',
       'Prime Video', 'Hulu', 'Netflix,CBS All Access,Hoopla',
       'HBO MAX,Hulu,TBS', 'Netflix,Comedy Central,fuboTV',
       'Free Services,Netflix,TNT', 'Free Services,Netflix,Hulu',
       'Free Services,Prime Video,NBC', 'Prime Video,USA,fuboTV',
       'Free Services,Netflix,YouTube Premium', 'Free Services,Hulu,FX',
       'Disney+', 'Prime Video,Hulu', 'Netflix,AMC,fuboTV',
       'Free Services,Hulu', 'Free Services,Hulu,TBS',
       'Free Services,Netflix,Showtime',
       'Free Services,Netflix,Prime Video', 'Free Services,Netflix',
       'Free Services', 'Netflix,Showtime,fuboTV',
       'Free Services,Hulu,ABC', 'Free Services,Prime Video',
       'Hulu,FX,Viceland', 'Netflix,Showtime,Hulu', 'Hulu,TBS,fuboTV',
       'Free Services,Netflix,HBO MAX',
       'Free Services,Prime Video,BritBox',
       'Free Services,Showtime,fuboTV', 'Free Services,Netflix,fuboTV',
       'Free Services,Hulu,Crunchyroll', 'Ne

In [22]:
platform_map = {
    'HBO MAX,HBO': 'HBO MAX',
    'Amazon Prime Video': 'Prime Video',
    'Prime': 'Prime Video',
    'Hulu Plus': 'Hulu',
    'Free Services': 'Free',
    'Disney+ Hotstar': 'Disney',
    'Disney+': 'Disney',
    'YouTube Premium': 'YouTube'
}

def clean_platforms(platform_entry):
    if pd.isna(platform_entry):
        return None  # Return None for missing values to later remove rows
    
    # Split platforms by comma and strip whitespace
    platforms = [p.strip() for p in platform_entry.split(',')]
    
    # Standardize platform names using the mapping
    platforms = [platform_map.get(p, p) for p in platforms]
    
    # Remove duplicates and return a sorted list of platforms
    return sorted(set(platforms))

In [23]:
# Apply the clean_platforms function to the 'platform' column
series_data_genre_cleaned['platform_cleaned'] = series_data_genre_cleaned['platform'].apply(clean_platforms)

# Remove rows where cleaned_platforms is None
series_data_genre_cleaned = series_data_genre_cleaned.dropna(subset=['platform_cleaned'])

/tmp/ipykernel_11833/1889650220.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  series_data_genre_cleaned['platform_cleaned'] = series_data_genre_cleaned['platform'].apply(clean_platforms)


In [24]:
series_data_genre_cleaned.sample(20)

,series_title,genres,description,platform,platform_cleaned
5378,The Opposition with Jordan Klepper,[Comedy],"A satire of the hyperbolic, conspiracy-laden n...","Free Services,Comedy Central","[Comedy Central, Free]"
1647,George Lopez,"[Comedy, Drama]",George Lopez is an American sitcom starring co...,Peacock Premium,[Peacock Premium]
9386,"Crazy, Lovely, Cool",[Romance],"CRAZY, LOVELY, COOL is a coming-of-age TV seri...",Netflix,[Netflix]
6390,Urara Meirocho,[Animation],"This is Meiro-machi (Labyrinth Town), the town...",Hulu,[Hulu]
670,Terra Nova,"[Action & Adventure, Science-Fiction]","In the year 2149, the world is dying. The plan...",Free Services,[Free]
2768,Relic Hunter,"[Fantasy, Mystery]",Relic Hunter is an anglophone Canadian televis...,Free Services,[Free]
9410,Fields of Gold,[Drama],A two-part conspiracy thriller about an eager ...,BritBox,[BritBox]
1642,Alcatraz,"[Action & Adventure, Crime]",A unique team investigates the shocking reappe...,Free Services,[Free]
5007,Tenchi Muyô! GXP,"[Animation, Comedy]","Seina is unlucky, so unlucky that when he stum...",Hulu,[Hulu]
7423,Love at First Kiss,[Reality],TLC throws out traditional ways of meeting som...,TLC,[TLC]


In [25]:
series_data_genre_cleaned.shape

(8404, 5)

In [26]:
# Display the unique genres
unique_platforms = set([genre for genres_list in series_data_genre_cleaned['platform_cleaned'] for genre in genres_list])
print(unique_platforms)

{'Viceland', 'Apple TV+', 'FYI', 'TruTV', 'NatGeo', 'BritBox', 'History', 'HGTV', 'Comedy Central', 'Hulu', 'Adult Swim', 'Travel Channel', 'Funimation', 'Lifetime', 'BET+', 'ABC', 'Free', 'Disney', 'IFC', 'AMC Premiere', 'Hoopla', 'Hallmark Movies Now', 'IndieFlix', 'DIY', 'Prime Video', 'TBS', 'USA', 'Netflix', 'Showtime', 'AMC', 'Peacock Premium', 'fuboTV', 'Crunchyroll', 'HBO MAX', 'Syfy', 'BBC America', 'FOX', 'Hallmark', 'YouTube', 'FX', 'MTV', 'Sundance', 'BET', 'CNBC', 'Cartoon Network', 'TVLand', 'TLC', 'AcornTV', 'Cinemax', 'Shudder', 'Food Network', 'Nick', 'VH1', 'CBS All Access', 'HBO', 'A&E', 'Bravo', 'Science', 'NBC', 'Starz', 'DC Universe', 'Epix', 'TNT'}


In [27]:
series_data_genre_cleaned.sample(20)

,series_title,genres,description,platform,platform_cleaned
1594,Alpha House,"[Comedy, Drama]",Four Republican senators share the same D.C. h...,Prime Video,[Prime Video]
2706,Green Acres,"[Comedy, Family]",Green Acres is an American sitcom starring Edd...,Prime Video,[Prime Video]
7023,Sell This House,[Reality],Sell This House is a reality television series...,"Free Services,FYI,A&E","[A&E, FYI, Free]"
4825,Ask the Storybots,"[Animation, Action & Adventure]","Based on the award-winning educational apps, t...",Netflix,[Netflix]
6964,My Ghost Story,"[Documentary, Reality]",Ordinary people reveal their terrifying experi...,fuboTV,[fuboTV]
5799,Killer Instinct with Chris Hansen,"[Documentary, Crime]",Chris Hansen brings his unrivaled journalistic...,Free Services,[Free]
6072,Teen Mom: Young + Pregnant,"[Reality, Drama]","Ashley, Brianna, Jade, Kayla and Lexi are five...",MTV,[MTV]
2930,The Riches,"[Drama, Comedy]",A family of crooks assume the identity of an u...,Hulu,[Hulu]
9684,Backroads USA,[Documentary],A 5-part series about five legendary roads in ...,"Free Services,Prime Video","[Free, Prime Video]"
5000,Archie's Weird Mysteries,"[Children, Animation]",Archie's Weird Mysteries is an American animat...,"Free Services,CBS All Access,Hoopla","[CBS All Access, Free, Hoopla]"


In [28]:
series_data_genre_platform_cleaned = series_data_genre_cleaned[['series_title', 'genres', 'platform_cleaned', 'description']]
series_data_genre_platform_cleaned.head()

,series_title,genres,platform_cleaned,description
0,Breaking Bad,"[Crime, Drama]",[Netflix],"When Walter White, a New Mexico chemistry teac..."
1,Game of Thrones,"[Action & Adventure, Drama]","[HBO, HBO MAX]",Seven noble families fight for control of the ...
2,Rick and Morty,"[Animation, Comedy]","[Free, HBO MAX, Hulu]",Rick is a mentally-unbalanced but scientifical...
3,Stranger Things,"[Drama, Fantasy]",[Netflix],"When a young boy vanishes, a small town uncove..."
4,The Boys,"[Action & Adventure, Comedy]",[Prime Video],A group of vigilantes known informally as “The...


In [29]:
series_data_genre_platform_cleaned.iloc[3].description

"When a young boy vanishes, a small town uncovers a mystery involving secret experiments, terrifying supernatural forces, and one strange little girl.Stranger Things featuring Winona Ryder and David Harbour has one or more episodes streaming with subscription on Netflix. It's an action & adventure and drama show with 25 episodes over 3 seasons. Stranger Things is still airing with no announced date for the next episode or season. It has a high IMDb audience rating of 8.8 (766,806 votes) and was very well received by critics."

In [30]:
# Cleaning steps
series_data_genre_platform_cleaned['description'] = series_data_genre_platform_cleaned['description'].str.strip()  # Remove leading/trailing spaces
series_data_genre_platform_cleaned['description'] = series_data_genre_platform_cleaned['description'].str.lower()  # Convert to lowercase

/tmp/ipykernel_11833/3908357255.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  series_data_genre_platform_cleaned['description'] = series_data_genre_platform_cleaned['description'].str.strip()  # Remove leading/trailing spaces
/tmp/ipykernel_11833/3908357255.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  series_data_genre_platform_cleaned['description'] = series_data_genre_platform_cleaned['description'].str.lower()  # Convert to lowercase


In [31]:
series_data_genre_platform_cleaned['description'] = series_data_genre_platform_cleaned['description'].str.replace(r'[^a-zA-Z0-9\s]', '', regex=True)  # Remove special characters
series_data_genre_platform_cleaned['description'] = series_data_genre_platform_cleaned['description'].str.replace(r'\s+', ' ', regex=True)  # Remove extra spaces

/tmp/ipykernel_11833/1524197337.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  series_data_genre_platform_cleaned['description'] = series_data_genre_platform_cleaned['description'].str.replace(r'[^a-zA-Z0-9\s]', '', regex=True)  # Remove special characters
/tmp/ipykernel_11833/1524197337.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  series_data_genre_platform_cleaned['description'] = series_data_genre_platform_cleaned['description'].str.replace(r'\s+', ' ', regex=True)  # Remove extra spaces


In [32]:
series_data_genre_platform_cleaned = series_data_genre_platform_cleaned.dropna(subset=['description'])
series_data_genre_platform_cleaned.shape

(8404, 4)

In [33]:
series_data_genre_platform_cleaned.iloc[3].description

'when a young boy vanishes a small town uncovers a mystery involving secret experiments terrifying supernatural forces and one strange little girlstranger things featuring winona ryder and david harbour has one or more episodes streaming with subscription on netflix its an action adventure and drama show with 25 episodes over 3 seasons stranger things is still airing with no announced date for the next episode or season it has a high imdb audience rating of 88 766806 votes and was very well received by critics'

In [34]:
series_data_genre_platform_cleaned["description"] = series_data_genre_platform_cleaned["description"].apply(lambda i: i.split())

In [35]:
series_data_genre_platform_cleaned.head()

,series_title,genres,platform_cleaned,description
0,Breaking Bad,"[Crime, Drama]",[Netflix],"[when, walter, white, a, new, mexico, chemistr..."
1,Game of Thrones,"[Action & Adventure, Drama]","[HBO, HBO MAX]","[seven, noble, families, fight, for, control, ..."
2,Rick and Morty,"[Animation, Comedy]","[Free, HBO MAX, Hulu]","[rick, is, a, mentallyunbalanced, but, scienti..."
3,Stranger Things,"[Drama, Fantasy]",[Netflix],"[when, a, young, boy, vanishes, a, small, town..."
4,The Boys,"[Action & Adventure, Comedy]",[Prime Video],"[a, group, of, vigilantes, known, informally, ..."


In [36]:
series_data_genre_platform_cleaned["tags"] = series_data_genre_platform_cleaned["genres"] + series_data_genre_platform_cleaned["platform_cleaned"] + series_data_genre_platform_cleaned["description"]

In [37]:
series_data_genre_platform_cleaned["tags"] = series_data_genre_platform_cleaned["tags"].apply(lambda  i: " ".join(i))

In [38]:
series_data_genre_platform_cleaned = series_data_genre_platform_cleaned.assign(id=range(1, len(series_data_genre_platform_cleaned) + 1))

In [39]:
cleaned_series_data = series_data_genre_platform_cleaned[["id", "series_title", "tags"]]

In [40]:
cleaned_series_data.sample(5)

,id,series_title,tags
7614,6441,Washington Heights,Reality BET+ follows a group of best friends l...
1987,1821,The Loud House,Animation Action & Adventure CBS All Access Fr...
5964,5132,La Patrona,Drama Hulu gabriela surez aracely armbula is t...
4702,4102,Ad Vitam,Crime Drama Netflix in a world where death is ...
10051,8038,All You Can Eat,Documentary Hoopla comedian john pinette sets ...


In [41]:
cleaned_series_data.iloc[3].tags

'Drama Fantasy Netflix when a young boy vanishes a small town uncovers a mystery involving secret experiments terrifying supernatural forces and one strange little girlstranger things featuring winona ryder and david harbour has one or more episodes streaming with subscription on netflix its an action adventure and drama show with 25 episodes over 3 seasons stranger things is still airing with no announced date for the next episode or season it has a high imdb audience rating of 88 766806 votes and was very well received by critics'

# Creating User Interest Data¶

In [42]:
users = pd.read_csv("users.csv")
users.head()

,user_id,user_name,email
0,1,Mohit,mohit07@gmail.com
1,2,Amitesh,amitesh03@gmail.com
2,3,Virat,virat18@gmail.com
3,4,Roman,roman01@gmail.com


In [43]:
series_data_genre_platform_cleaned.sample(5)

,series_title,genres,platform_cleaned,description,tags,id
9740,Stephen Tompkinson's Australian Balloon Adventure,"[Documentary, Action & Adventure]","[Free, Prime Video]","[stephen, tompkinsons, adventure, begins, in, ...",Documentary Action & Adventure Free Prime Vide...,7892
9941,Regalia: The Three Sacred Stars,"[Animation, Action & Adventure]",[Funimation],"[12, years, ago, in, the, country, of, rimguar...",Animation Action & Adventure Funimation 12 yea...,7990
7619,After School Dice Club,"[Animation, Comedy]","[Funimation, Hulu]","[a, story, about, girls, playing, board, games...",Animation Comedy Funimation Hulu a story about...,6445
6690,"Three Wives, One Husband",[Documentary],[Netflix],"[documentary, providing, access, to, the, comm...",Documentary Netflix documentary providing acce...,5715
7211,Starhunter ReduX,"[Action & Adventure, Science-Fiction]",[Prime Video],"[the, story, of, the, 44, episodes, of, starhu...",Action & Adventure Science-Fiction Prime Video...,6122


In [44]:
# Display the unique genres

unique_genres = set([genre for genres_list in series_data_genre_platform_cleaned['genres'] for genre in genres_list])
print(unique_genres)

{'Romance', 'Documentary', 'Action & Adventure', 'Crime', 'Biography', 'Drama', 'Science-Fiction', 'Thriller', 'Fantasy', 'Reality', 'Horror', 'Sport', 'Family', 'Children', 'Game Show', 'Mystery', 'Comedy', 'Musical', 'Animation', 'Stand-up & Talk'}


In [45]:
user_interests = {
    1: ['Mystery', 'Comedy', 'Romance', 'ScienceFiction', 'Drama'],                       # Mohit
    2: ['Crime', 'Action & Adventure', 'Thriller', 'Musical'],                            # Amitesh
    3: ['Horror', 'Animation', 'Action & Adventure', 'Mystery', 'Action & Adventure'],    # Virat
    4: ['Documentary', 'Family', 'Comedy', 'Romance']                                     # Roman
}

In [46]:
# Parameters
num_users = 4
num_movies = 8404
num_interactions = 256  # Desired number of interactions

# Helper function to get movies with the most genre matches
def get_best_matching_series(user_id, series):
    interests = user_interests[user_id]
    # print("interests", interests)
    # Calculate the number of matching genres for each movie
    series['genre_match_count'] = series['genres'].apply(
        lambda genres: len(set(genres).intersection(set(interests)))
    )
    # print("series", series['genre_match_count'])
    
    # Filter movies with at least 2 matching genre and sort by match count
    matching_series = series[series['genre_match_count'] >=1].sort_values(by='genre_match_count', ascending=False)
    # print("matching_series", matching_series)
    
    # Return the list of movie IDs sorted by the highest genre match count
    return matching_series['id'].tolist()

# Generate Interactions
interactions = []

for _ in range(num_interactions):
    user_id = random.choice(users['user_id'])
    # Get the best matching movies for the user
    best_matching_series_ids = get_best_matching_series(user_id, series_data_genre_platform_cleaned)
    # print("best_matching_series_ids", best_matching_series_ids)
    if not best_matching_series_ids:
        continue  # Skip if no matching movies found
    series_id = random.choice(best_matching_series_ids)
    rating = random.choice([1, 2, 3, 4, 5])  # Random rating from 1 to 5

    # Add interaction
    interactions.append({
        'user_id': user_id,
        'id': series_id,
        'rating': rating
    })

# Create Interaction DataFrame
interaction_df = pd.DataFrame(interactions)

# Optionally, reset the index
interaction_df.reset_index(drop=True, inplace=True)

In [47]:
interaction_df.sample(10)

,user_id,id,rating
253,1,5484,4
1,4,2670,5
37,4,6154,1
108,2,5180,1
216,2,8010,1
181,3,2830,3
250,1,4154,2
234,1,1470,1
164,2,6159,1
5,2,331,1


In [48]:
merged_df = pd.merge(interaction_df, series_data_genre_platform_cleaned, on='id', how='left')
merged_df.head(5)

,user_id,id,rating,series_title,genres,platform_cleaned,description,tags,genre_match_count
0,2,2097,5,Infinite Stratos,"[Action & Adventure, Animation]","[Free, Hulu]","[in, the, near, future, a, japanese, scientist...",Action & Adventure Animation Free Hulu in the ...,0
1,4,2670,5,Hangar 1: The UFO Files,"[Documentary, Mystery]","[Hulu, Netflix, Prime Video]","[delve, deep, into, a, vast, archive, of, over...",Documentary Mystery Hulu Netflix Prime Video d...,1
2,3,4789,4,"Philip Marlowe, Private Eye","[Action & Adventure, Crime]",[Prime Video],"[philip, marlowe, private, eye, is, a, british...",Action & Adventure Crime Prime Video philip ma...,0
3,4,5781,4,The African Americans: Many Rivers to Cross wi...,[Documentary],"[Hoopla, Prime Video]","[professor, gates, describes, the, history, of...",Documentary Hoopla Prime Video professor gates...,1
4,4,1087,4,Kim's Convenience,[Comedy],[Netflix],"[the, funny, heartfelt, story, of, the, kims, ...",Comedy Netflix the funny heartfelt story of th...,1


In [49]:
user_1_interactions = merged_df[merged_df['user_id'] == 1]
print(user_1_interactions.shape)
user_1_interactions.sample(20)

(66, 9)


,user_id,id,rating,series_title,genres,platform_cleaned,description,tags,genre_match_count
213,1,2760,1,Day And Night,"[Crime, Drama]",[Netflix],"[the, usually, carefree, guan, hong, yu, is, a...",Crime Drama Netflix the usually carefree guan ...,0
86,1,2393,2,Twenties,[Comedy],"[BET, Free, Showtime]","[hattie, a, queer, african, american, woman, h...",Comedy BET Free Showtime hattie a queer africa...,1
90,1,2662,1,Benched,[Comedy],[Free],"[nina, is, a, dedicated, career, driven, corpo...",Comedy Free nina is a dedicated career driven ...,1
159,1,3322,3,Once Upon a Time in Lingjian Mountain,"[Action & Adventure, Comedy]",[Netflix],"[a, story, that, follows, wang, lu, a, young, ...",Action & Adventure Comedy Netflix a story that...,1
219,1,5772,2,Wake Up,[Drama],[Netflix],"[wake, up, featuring, lego, li, and, tiffany, ...",Drama Netflix wake up featuring lego li and ti...,0
41,1,685,2,Treme,"[Drama, Musical]","[HBO, HBO MAX]","[trem, takes, its, name, from, a, neighborhood...",Drama Musical HBO HBO MAX trem takes its name ...,0
69,1,3033,1,Ransom,"[Drama, Action & Adventure]",[CBS All Access],"[eric, beaumonts, crisis, negotiator, team, is...",Drama Action & Adventure CBS All Access eric b...,0
231,1,5188,2,Public Enemies,"[Drama, Crime]",[Prime Video],"[public, enemies, explores, the, relationship,...",Drama Crime Prime Video public enemies explore...,0
146,1,2233,4,Upstairs Downstairs,"[Drama, Romance]","[BritBox, Hulu]","[set, in, 1936, the, show, takes, viewers, old...",Drama Romance BritBox Hulu set in 1936 the sho...,1
163,1,1712,2,House of Cards,[Drama],[Free],"[charming, chief, whip, frances, urquhart, plo...",Drama Free charming chief whip frances urquhar...,0


In [50]:
interaction_df.to_csv('series_interaction_df.csv', index=False)

# Model Building

In [51]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import pickle

In [52]:
users_interest_data = pd.read_csv('series_interaction_df.csv')

In [53]:
users.head()

,user_id,user_name,email
0,1,Mohit,mohit07@gmail.com
1,2,Amitesh,amitesh03@gmail.com
2,3,Virat,virat18@gmail.com
3,4,Roman,roman01@gmail.com


In [54]:
users_interest_data.head()

,user_id,id,rating
0,2,2097,5
1,4,2670,5
2,3,4789,4
3,4,5781,4
4,4,1087,4


In [55]:
series_data_genre_platform_cleaned.head()

,series_title,genres,platform_cleaned,description,tags,id,genre_match_count
0,Breaking Bad,"[Crime, Drama]",[Netflix],"[when, walter, white, a, new, mexico, chemistr...",Crime Drama Netflix when walter white a new me...,1,0
1,Game of Thrones,"[Action & Adventure, Drama]","[HBO, HBO MAX]","[seven, noble, families, fight, for, control, ...",Action & Adventure Drama HBO HBO MAX seven nob...,2,0
2,Rick and Morty,"[Animation, Comedy]","[Free, HBO MAX, Hulu]","[rick, is, a, mentallyunbalanced, but, scienti...",Animation Comedy Free HBO MAX Hulu rick is a m...,3,1
3,Stranger Things,"[Drama, Fantasy]",[Netflix],"[when, a, young, boy, vanishes, a, small, town...",Drama Fantasy Netflix when a young boy vanishe...,4,0
4,The Boys,"[Action & Adventure, Comedy]",[Prime Video],"[a, group, of, vigilantes, known, informally, ...",Action & Adventure Comedy Prime Video a group ...,5,1


In [56]:
cv = CountVectorizer(max_features=5000, stop_words="english")
tfd = TfidfVectorizer(max_features=3000)

In [57]:
vector = cv.fit_transform(cleaned_series_data["tags"]).toarray()
vector

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [58]:
# FINDING THE DISTANCES BETWEEN THE MOVIES
similarity = cosine_similarity(vector)
similarity

array([[1.        , 0.43147295, 0.26039606, ..., 0.25007089, 0.14182079,
        0.14975062],
       [0.43147295, 1.        , 0.45046302, ..., 0.18128057, 0.15180282,
        0.28496141],
       [0.26039606, 0.45046302, 1.        , ..., 0.18182152, 0.39699293,
        0.28792888],
       ...,
       [0.25007089, 0.18128057, 0.18182152, ..., 1.        , 0.19970314,
        0.28967923],
       [0.14182079, 0.15180282, 0.39699293, ..., 0.19970314, 1.        ,
        0.27904046],
       [0.14975062, 0.28496141, 0.28792888, ..., 0.28967923, 0.27904046,
        1.        ]])

In [59]:
sorted(list(enumerate(similarity[0])), reverse=True, key=lambda x: x[1])[1:6]

[(116, 0.6902114351624684),
 (754, 0.6782619741670973),
 (1577, 0.6436544669061126),
 (669, 0.6430394361098801),
 (533, 0.6403729533481601)]

In [60]:
# Calculate the average rating and number of interactions for each product
popularity_data = users_interest_data.groupby('id').agg(
    average_rating=('rating', 'mean'),
    num_interactions=('id', 'count')
).reset_index()

# Merge popularity data with the products data
cleaned_series_data = cleaned_series_data.merge(popularity_data, on='id', how='left')

# Fill NaN values (if any) with 0
# Normalize the average_rating and num_interactions using MinMaxScaler
scaler = MinMaxScaler()
cleaned_series_data[['normalized_rating', 'normalized_interactions']] = scaler.fit_transform(
    cleaned_series_data[['average_rating', 'num_interactions']]
)

# Calculate a combined popularity score
cleaned_series_data['num_interactions'].fillna(0, inplace=True)

/tmp/ipykernel_11833/1960803139.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  cleaned_series_data['num_interactions'].fillna(0, inplace=True)


In [61]:
cleaned_series_data.sample(10)

,id,series_title,tags,average_rating,num_interactions,normalized_rating,normalized_interactions
3806,3807,Home Game,Documentary Sport Netflix this docuseries prof...,NaN,0.0,NaN,NaN
4296,4297,The Ruth Rendell Mysteries,Drama Crime AcornTV BritBox Prime Video the ru...,NaN,0.0,NaN,NaN
382,383,See,Drama Science-Fiction Apple TV+ a virus has de...,NaN,0.0,NaN,NaN
4025,4026,Kimba the White Lion,Action & Adventure Animation Funimation the ad...,NaN,0.0,NaN,NaN
7923,7924,Ochocinco: The Ultimate Catch,Reality Free ochocinco the ultimate catch is a...,NaN,0.0,NaN,NaN
3326,3327,Carol's Second Act,Comedy CBS All Access Free after raising her t...,NaN,0.0,NaN,NaN
2667,2668,Intersection,Drama Netflix naz who is a paediatrician loses...,NaN,0.0,NaN,NaN
4650,4651,New Worlds,Action & Adventure Drama AcornTV Hoopla new wo...,NaN,0.0,NaN,NaN
1196,1197,The Guild,Comedy Drama Netflix the guild is an american ...,NaN,0.0,NaN,NaN
2623,2624,The Odd Couple,Comedy CBS All Access Hulu oscars life seems a...,3.0,1.0,0.5,0.0


In [62]:
cleaned_series_data['num_interactions'].unique()

array([0., 1., 2.])

In [63]:
# Step 1: Fill NaN values in the 'num_interactions' column with 0 (if there are any)
cleaned_series_data['num_interactions'].fillna(0, inplace=True)

# Step 2: Normalize 'average_rating' and 'num_interactions' using MinMaxScaler
scaler = MinMaxScaler()

# Normalize 'average_rating' and 'num_interactions' columns
cleaned_series_data[['normalized_rating', 'normalized_interactions']] = scaler.fit_transform(
    cleaned_series_data[['average_rating', 'num_interactions']]
)

# Step 3: Calculate the combined popularity score
cleaned_series_data['popularity_score'] = (cleaned_series_data['normalized_rating'] + cleaned_series_data['normalized_interactions']) / 2

# Step 4: Fill any NaN values in 'popularity_score' (if any)
cleaned_series_data['popularity_score'].fillna(0, inplace=True)

/tmp/ipykernel_11833/1257570391.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  cleaned_series_data['num_interactions'].fillna(0, inplace=True)
/tmp/ipykernel_11833/1257570391.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, in

In [64]:
cleaned_series_data.head()

,id,series_title,tags,average_rating,num_interactions,normalized_rating,normalized_interactions,popularity_score
0,1,Breaking Bad,Crime Drama Netflix when walter white a new me...,NaN,0.0,NaN,0.0,0.0
1,2,Game of Thrones,Action & Adventure Drama HBO HBO MAX seven nob...,NaN,0.0,NaN,0.0,0.0
2,3,Rick and Morty,Animation Comedy Free HBO MAX Hulu rick is a m...,NaN,0.0,NaN,0.0,0.0
3,4,Stranger Things,Drama Fantasy Netflix when a young boy vanishe...,NaN,0.0,NaN,0.0,0.0
4,5,The Boys,Action & Adventure Comedy Prime Video a group ...,NaN,0.0,NaN,0.0,0.0


In [65]:
cleaned_series_data['popularity_score'].unique()

array([0.    , 0.375 , 0.75  , 0.625 , 0.25  , 0.8125, 0.5   , 0.875 ,
       0.9375])

In [66]:
def recommend_series_for_user(user_id, users_interest_data, movie_data, similarity_matrix, top_n=10, threshold=0.24):
    # Get the list of movie IDs the user is interested in (assuming interest is based on 'movie_id')
    user_interests = users_interest_data[users_interest_data['user_id'] == user_id]['id'].tolist()
    # print(user_interests)

    # Check if the user has any interests (if the user is new and has no interests)
    if not user_interests:
        print("New user detected. Providing popularity-based recommendations.")
        
        # Sort movies based on popularity score or other metric (assuming popularity_score exists)
        popular_movies = movie_data.sort_values(by='popularity_score', ascending=False)
        
        # Return the top N popular movies
        return popular_movies['series_title'].head(top_n).tolist()

    # Get the indices of the user's interested movies in the movie_data DataFrame
    movie_indices = [movie_data[movie_data['id'] == movie_id].index[0] for movie_id in user_interests]
    # print(user_interests)

    # Calculate the average similarity score for each movie based on the user's interests
    # similarity_matrix: Movie similarity scores between movies
    similarity_scores = sum(similarity_matrix[idx] for idx in movie_indices) / len(movie_indices)

    # Sort movies based on similarity scores (high to low)
    movie_list = sorted(list(enumerate(similarity_scores)), key=lambda x: x[1], reverse=True)
    # print(movie_list)
    

    # Filter out movies that the user has already interacted with and apply the threshold for similarity
    recommended_movies = [
        movie_data.iloc[i[0]]['series_title'] for i in movie_list
        if movie_data.iloc[i[0]]['id'] not in user_interests and i[1] >threshold
    ]

    # Return the top N recommended movies
    print(f"Recommending {len(recommended_movies)} movies to user {user_id}")
    return recommended_movies


In [76]:
recommended_series = recommend_series_for_user(1, users_interest_data, cleaned_series_data, similarity)
# print("Recommended Series:", recommended_series)

Recommending 7373 movies to user 1


In [69]:
recommended_movie_details = cleaned_series_data[cleaned_series_data['series_title'].isin(recommended_series)]
recommended_movie_details.shape

(7387, 8)

In [70]:
def split_user_interests(users_interest_data, user_id):
    user_interests = users_interest_data[users_interest_data['user_id'] == user_id]['id'].tolist()
    train_interests, test_interests = train_test_split(user_interests, test_size=0.3, random_state=42)
    return train_interests, test_interests

In [71]:
def evaluated_recommend_products_for_user(user_id, users_interest_data, products_data, similarity_matrix):
    # Split user interests into training and testing
    train_interests, test_interests = split_user_interests(users_interest_data, user_id)
    
    # Get the indices of these products in the products_data DataFrame
    product_indices = [products_data[products_data['id'] == pid].index[0] for pid in train_interests]

    # Calculate the average similarity score for each product based on the user's training interests
    similarity_scores = sum(similarity_matrix[idx] for idx in product_indices) / len(product_indices)

    # Sort products based on similarity scores in descending order
    product_list = sorted(list(enumerate(similarity_scores)), key=lambda x: x[1], reverse=True)

    threshold = 0.24
    recommended_products = [
        products_data.iloc[i[0]]['id'] for i in product_list
        if products_data.iloc[i[0]]['id'] not in train_interests and i[1] > threshold
    ]

    # Return the top 15 recommended product IDs
    return recommended_products[:10], test_interests


In [72]:
from sklearn.metrics import precision_score, recall_score, f1_score

def evaluate_recommendations(recommended_products, test_interests):
    y_true = [1 if pid in test_interests else 0 for pid in recommended_products]
    y_pred = [1] * len(recommended_products)

    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print(f"Precision: {precision:.2f}")
    print(f"Recall: {recall:.2f}")
    print(f"F1-Score: {f1:.2f}")

In [73]:
recommended_movies, test_interests = evaluated_recommend_products_for_user(1, users_interest_data, cleaned_series_data, similarity)
evaluate_recommendations(recommended_movies, test_interests)

Precision: 0.00
Recall: 0.00
F1-Score: 0.00


/home/mohit/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [74]:
pickle.dump(cleaned_series_data, open("series_data.pkl","wb"))

In [75]:
pickle.dump(similarity, open("series_similarity_matrix.pkl", "wb"))